In [12]:
# =========================================
# 0) Imports, paths, and helpers
# =========================================
import numpy as np
import pandas as pd
from pathlib import Path

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.dummy import DummyClassifier
from packaging import version
from packaging import version

# ---------- Paths (adjust if needed) ----------
DATA_PATH = Path("samadult.csv")       # raw NHIS Sample Adult file (742 vars)
OUT_DIR   = Path("prep_outputs")       # all outputs will be saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Small utility: safe list writer ----------
def write_list(lst, path: Path):
    """Write a list to a txt file, one item per line, no header/index."""
    pd.Series(lst).to_csv(path, index=False, header=False)

In [13]:
# =========================================
# 1) Load data + global config
# =========================================
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", df_raw.shape)

# Target and survey design/ID/weights to be excluded from predictors
TARGET = "AMIGR"
DESIGN_ID = [c for c in ["HHX","FMX","FPX","WTIA_SA","WTFA_SA","PSTRAT","PPSU","SRVY_YR","RECTYPE","INTV_QRT","INTV_MON"]
             if c in df_raw.columns]
BAN = set(DESIGN_ID + [TARGET])

print("Banned (design/ID/weights + target):", sorted(BAN))

Raw shape: (25417, 742)
Banned (design/ID/weights + target): ['AMIGR', 'FMX', 'FPX', 'HHX', 'INTV_MON', 'INTV_QRT', 'PPSU', 'PSTRAT', 'RECTYPE', 'SRVY_YR', 'WTFA_SA', 'WTIA_SA']


In [14]:
# =========================================
# 2) Constant variables (raw) + effective missingness
# =========================================

# 2.1 Strict constants on raw data (NaN not counted as a unique value)
#     dropna=True -> consistent with Iteration 2/SPSS style
nunique_raw_dropna = df_raw.nunique(dropna=True)
constant_vars_raw  = nunique_raw_dropna[nunique_raw_dropna == 1].index.tolist()

print("Strict constants (dropna=True):", len(constant_vars_raw))
write_list(constant_vars_raw, OUT_DIR/"constant_vars.txt")

# Also report constants when NaN is counted as a value
nunique_raw_keepna = df_raw.nunique(dropna=False)
constant_vars_keepna = nunique_raw_keepna[nunique_raw_keepna == 1].index.tolist()
print("Strict constants (dropna=False):", len(constant_vars_keepna))

# 2.2 Effective missingness: map NHIS nonresponse codes to NaN
#     and compute missing ratio per column on-the-fly.
BAD_CODES_GLOBAL = {7, 8, 9, 97, 98, 99, 997, 998, 999, 9999}

def effective_na_series_QC(s: pd.Series, name: str) -> pd.Series:
    """
    Lightweight mapper for 3.1 ONLY:
    - Convert to numeric so "99" -> 99
    - Replace common NHIS nonresponse codes with NaN
    - Minimal special handling 
    """
    x = pd.to_numeric(s, errors="coerce")

    return x.replace({code: np.nan for code in BAD_CODES_GLOBAL})

def effective_missing_rate(s: pd.Series, name: str) -> float:
    x = effective_na_series_QC(s, name)
    return float(x.isna().mean())

eff_miss = pd.Series({c: effective_missing_rate(df_raw[c], c) for c in df_raw.columns}, name="missing_eff")

# Save a full quality table (raw uniques + effective missing)
quality_tbl = pd.DataFrame({
    "unique_raw_dropna": nunique_raw_dropna,
    "unique_raw_keepna": nunique_raw_keepna,
    "missing_eff": eff_miss
}).sort_values("missing_eff", ascending=False)

quality_tbl.to_csv(OUT_DIR/"quality_summary_all_vars.csv")
quality_tbl.head(10)

Strict constants (dropna=True): 24
Strict constants (dropna=False): 2


,unique_raw_dropna,unique_raw_keepna,missing_eff
ALDURA91,2,3,0.999961
ALTIME91,2,3,0.999961
ALDURB91,1,2,0.999921
ALCHRC91,1,2,0.999921
ALDURA29,3,4,0.999921
ALTIME29,3,4,0.999921
ALUNIT91,1,2,0.999921
ALUNIT29,1,2,0.999882
ALCHRC29,1,2,0.999882
ALDURB29,1,2,0.999882


In [15]:
# =========================================
# 3) Auto-build variable pools (hq_pool & domain_vars)
# =========================================
# If you already have curated lists, paste them below. Otherwise we auto-build.

# 3.1 Optional: paste a curated domain list here (leave None to start empty)
# domain_vars_seed = None
domain_vars_seed = ["ASISLEEP","BMI","SMKSTAT2","DOINGLWA","HYPEV","CHDEV","MIEV","HRTEV",
                    "CANEV","AHAYFYR","PAINFACE","PAINLB","DEP_1","ANX_1","TIRED_1",
                    "SEX","AGE_P","REGION","HISPAN_I","RACERPI2","R_MARITL", "HVPEV","ASPONOWN","DIBPRE2","SMKNOW","AWORPAY","APX12_1","ARX12_2","ARX12_3","ASIMEDC","ASIMUCH","PAIN_2A"]

# 3.2 Domain vars (clean up against BAN & existing columns)
if domain_vars_seed:
    domain_vars = [c for c in domain_vars_seed if c in df_raw.columns and c not in BAN]
else:
    domain_vars = []  # can be filled later or remain empty

# 3.3 Auto-build high-quality pool from df_raw (not banned, not constant, missing_eff <= threshold)
MISSING_TH = 0.40  
hq_pool = [c for c in df_raw.columns
           if c not in BAN
           and c not in set(constant_vars_raw)
           and eff_miss[c] <= MISSING_TH]

print(f"domain_vars: {len(domain_vars)} | hq_pool: {len(hq_pool)} | constants dropped: {len(constant_vars_raw)}")

# Save both lists for reproducibility
write_list(domain_vars, OUT_DIR/"domain_vars.txt")
write_list(hq_pool,     OUT_DIR/"hq_pool_vars.txt")

domain_vars: 30 | hq_pool: 225 | constants dropped: 24


In [16]:
# =========================================
# 4) Modelling table for feature selection
# =========================================
# Keep only target + hq_pool (already excludes IDs/weights)
df_use = df_raw[[TARGET] + hq_pool].copy()

# Keep only valid target codes {1,2}; encode y as 0/1
df_use = df_use[df_use[TARGET].isin([1,2])].copy()
y = (df_use[TARGET] == 1).astype(int)
X = df_use.drop(columns=[TARGET])

# Apply effective NA mapping used for quality (ensure consistency with modelling)
X_clean = pd.DataFrame({c: effective_na_series_QC(X[c], c) for c in X.columns})

# Numeric vs categorical split (on cleaned data)
num_cols = [c for c in X_clean.columns
            if pd.api.types.is_numeric_dtype(X_clean[c]) and X_clean[c].nunique(dropna=True) > 30]
cat_cols = sorted([c for c in X_clean.columns if c not in num_cols])

print(f"#num={len(num_cols)} | #cat={len(cat_cols)} | pool={len(hq_pool)}")

#num=17 | #cat=208 | pool=225


In [17]:
# =========================================
# 5) Preprocess pipeline + Random Forest
# =========================================
# OHE compatibility (sklearn >=1.2 uses sparse_output)
ohe_kwargs = {"handle_unknown": "ignore"}
if version.parse(sklearn.__version__) >= version.parse("1.2"):
    ohe_kwargs["sparse_output"] = False
else:
    ohe_kwargs["sparse"] = False

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median"))
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(**ohe_kwargs))
])

pre = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    min_samples_split=200,
    min_samples_leaf=30,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

pipe = Pipeline([
    ("pre", pre),
    ("clf", rf)
])

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)
print("RF train score:", pipe.score(X_train, y_train))
print("RF  test score:", pipe.score(X_test,  y_test))

y_pred  = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print("\n--- Classification report (test) ---")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion matrix (test):")
print(confusion_matrix(y_test, y_pred))

print("ROC AUC (test):", roc_auc_score(y_test, y_proba))

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
print("\nBaseline (most_frequent) acc:", dummy.score(X_test, y_test))

RF train score: 0.7753173900206672
RF  test score: 0.7482778980515646

--- Classification report (test) ---
              precision    recall  f1-score   support

           0      0.931     0.759     0.836      4307
           1      0.339     0.689     0.455       774

    accuracy                          0.748      5081
   macro avg      0.635     0.724     0.645      5081
weighted avg      0.841     0.748     0.778      5081

Confusion matrix (test):
[[3269 1038]
 [ 241  533]]
ROC AUC (test): 0.8003958461947349

Baseline (most_frequent) acc: 0.8476677819326904


In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_cv = cross_val_score(pipe, X_clean, y, cv=cv, scoring="roc_auc", n_jobs=-1)
print("\n5-fold ROC AUC (mean ± std):", f"{auc_cv.mean():.3f} ± {auc_cv.std():.3f}")

f1_cv = cross_val_score(pipe, X_clean, y, cv=cv, scoring="f1", n_jobs=-1)
print("5-fold F1 (mean ± std):", f"{f1_cv.mean():.3f} ± {f1_cv.std():.3f}")


5-fold ROC AUC (mean ± std): 0.807 ± 0.006
5-fold F1 (mean ± std): 0.464 ± 0.010


In [19]:
print("Label balance (y):")
print(y.value_counts(normalize=True).rename({0:"No-migraine", 1:"Migraine"}))

Label balance (y):
AMIGR
No-migraine    0.847656
Migraine       0.152344
Name: proportion, dtype: float64


In [23]:
# =========================================
# 6) Aggregate importances to base variables + save CLEAN Top-N
# =========================================
# Get feature names after preprocessing
if len(cat_cols):
    ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    cat_feature_names = ohe.get_feature_names_out(cat_cols)
else:
    cat_feature_names = np.array([])

num_feature_names = np.array(num_cols)
all_feature_names = np.concatenate([num_feature_names, cat_feature_names])

importances = pipe.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": all_feature_names, "importance": importances}) \
         .sort_values("importance", ascending=False)

def to_base(feat: str) -> str:
    """Map one-hot feature back to its raw base variable name."""
    for raw in cat_cols:
        if feat.startswith(raw + "__") or feat.startswith(raw + "_"):
            return raw
    return feat  # numeric or no split

imp_df["base"] = imp_df["feature"].map(to_base)
imp_agg = imp_df.groupby("base", as_index=False)["importance"].sum() \
                .sort_values("importance", ascending=False)

# Save full aggregated importances (appendix)
imp_agg.to_csv(OUT_DIR/"rf_feature_importance_aggregated.csv", index=False)

# Clean by removing BAN before writing Top-N
TOP_N = 30
imp_agg_clean = imp_agg[~imp_agg["base"].isin(BAN)].copy()
top_vars_data_driven = imp_agg_clean.head(TOP_N)["base"].tolist()

write_list(top_vars_data_driven, OUT_DIR/"top_vars_data_driven.txt")
print(f"Saved CLEAN data-driven top vars (N={len(top_vars_data_driven)}): prep_outputs/top_vars_data_driven.txt")
imp_agg_clean.head(10)

Saved CLEAN data-driven top vars (N=30): prep_outputs/top_vars_data_driven.txt


,base,importance
176,PAINECK,0.084943
178,PAINLB,0.074187
180,PAIN_4,0.048524
177,PAINFACE,0.033919
208,TIRED_3,0.033074
3,AGE_P,0.031862
179,PAIN_2A,0.030876
207,TIRED_2,0.028758
47,ANX_1,0.026427
100,BEDDAYR,0.025940


In [24]:
# Load quality summary
quality_summary_all_vars = pd.read_csv(OUT_DIR/"quality_summary_all_vars.csv", index_col=0)

# ========== Helper function ==========
def get_top_vars_by_threshold(threshold=0.3, top_k=30):
    # 1. Drop high-missing vars based on threshold
    eff_missing = quality_summary_all_vars["missing_eff"]  
    vars_keep = eff_missing[eff_missing <= threshold].index.tolist()
    
    # 2. Build a feature matrix 
    features = [c for c in vars_keep if c not in DESIGN_ID + [TARGET]]
    X = df_raw[features].copy()
    y = (df_raw[TARGET]==1).astype(int)

    # 3. Apply effective NA mapping
    X_clean = pd.DataFrame({c: effective_na_series_QC(X[c], c) for c in X.columns})

    # 4. Numeric / categorical split
    num_cols = [c for c in X_clean.columns
                if pd.api.types.is_numeric_dtype(X_clean[c]) and X_clean[c].nunique(dropna=True) > 30]
    cat_cols = sorted([c for c in X_clean.columns if c not in num_cols])

    # 5. Preprocessing
    ohe_kwargs = {"handle_unknown": "ignore"}
    if version.parse(sklearn.__version__) >= version.parse("1.2"):
        ohe_kwargs["sparse_output"] = False
    else:
        ohe_kwargs["sparse"] = False

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(**ohe_kwargs))
        ]), cat_cols)
    ])

    rf = RandomForestClassifier(
        n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=42
    )

    pipe = Pipeline([
        ("pre", pre),
        ("clf", rf)
    ])
    pipe.fit(X_clean, y)

    # 6. Importance aggregation
    ohe: OneHotEncoder = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    cat_feature_names = ohe.get_feature_names_out(cat_cols) if len(cat_cols) > 0 else np.array([])
    all_feature_names = np.concatenate([num_cols, cat_feature_names])

    imp_df = pd.DataFrame({
        "feature": all_feature_names,
        "importance": pipe.named_steps["clf"].feature_importances_
    })

    imp_df["base"] = imp_df["feature"].map(lambda f: f.split("__")[0].split("_")[0])
    imp_agg = imp_df.groupby("base", as_index=False)["importance"].sum().sort_values("importance", ascending=False)

    return set(imp_agg.head(top_k)["base"].tolist())

# ========== Run with 3 thresholds ==========
top_20 = get_top_vars_by_threshold(0.2, top_k=30)
top_30 = get_top_vars_by_threshold(0.3, top_k=30)
top_40 = get_top_vars_by_threshold(0.4, top_k=30)

# ========== Compute Jaccard similarities ==========
def jaccard(a, b): return len(a & b) / len(a | b)

print("Jaccard(20%,30%) =", jaccard(top_20, top_30))
print("Jaccard(30%,40%) =", jaccard(top_30, top_40))
print("Jaccard(20%,40%) =", jaccard(top_20, top_40))

Jaccard(20%,30%) = 0.8181818181818182
Jaccard(30%,40%) = 0.875
Jaccard(20%,40%) = 0.7647058823529411


In [25]:
# =========================================
# 7) Final feature set = union(data-driven, domain) minus BAN
# =========================================
final_vars = sorted((set(top_vars_data_driven) | set(domain_vars)) - set(BAN))
print("Final features count:", len(final_vars))

# Build final modelling table: RID (optional), TARGET, final_vars
df_final = df_raw.loc[:, [TARGET] + final_vars].copy()
df_final.insert(0, "RID", range(1, len(df_final) + 1))  # optional row ID for traceability

# Sanity check: ensure no design/ID/weights leakage
leak = set(df_final.columns) & set(DESIGN_ID)
print("Design/ID leakage ->", leak if leak else "No leakage ✅")

# Export
final_csv = OUT_DIR/"final_selected_table.csv"
df_final.to_csv(final_csv, index=False)
print("Saved final table:", final_csv, "| shape:", df_final.shape)

Final features count: 52
Design/ID leakage -> No leakage ✅
Saved final table: prep_outputs/final_selected_table.csv | shape: (25417, 54)
